In [1]:
import ray
from ray import serve
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import pyarrow as pa
import requests
from starlette.requests import Request
import json
from sentence_transformers import SentenceTransformer
import textdistance
import numpy as np

# Model Inference for Batch and Online Use Cases with Ray Data and Ray Serve

## Batch inference

Typical task: for each user in a dataset of users, find products to recommend, generate an email with those recommendations, and write out to storage/mail queue/DB

## Online inference

Typical tasks:

* Given a live user session, generate a recommendation
* Given a live user session and a search query, generate a product search result *and* a product recommendation

We will implement these use cases

We'll use an existing recommender model snapshot for these activities

In [2]:
! aws s3 sync s3://anyscale-public-materials-use2/ecom /mnt/cluster_storage/ecom

download: s3://anyscale-public-materials-use2/ecom/catalog/27_00db830ac2654c5f8012007929524983_000001_000000-0.parquet to ../../../mnt/cluster_storage/ecom/catalog/27_00db830ac2654c5f8012007929524983_000001_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/catalog/27_00db830ac2654c5f8012007929524983_000000_000000-0.parquet to ../../../mnt/cluster_storage/ecom/catalog/27_00db830ac2654c5f8012007929524983_000000_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/cat_with_embeddings/57_52effbfc1db2476c9c27d26ae7a15c12_000003_000000-0.parquet to ../../../mnt/cluster_storage/ecom/cat_with_embeddings/57_52effbfc1db2476c9c27d26ae7a15c12_000003_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/catalog/27_00db830ac2654c5f8012007929524983_000002_000000-0.parquet to ../../../mnt/cluster_storage/ecom/catalog/27_00db830ac2654c5f8012007929524983_000002_000000-0.parquet
download: s3://anyscale-public-materials-use2/ecom/cat_with_embeddings/57_52effb

In [ ]:
base_model_path = '/mnt/cluster_storage/ecom/recommender/base_model/model.pt'

## Batch inference

Batch inference is typically implemented with Ray Data pipelines

Here is our synthetic user dataset

In [ ]:
! head -29 /mnt/cluster_storage/ecom/users.json

In [ ]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.json')

This dataset is small, so we can materialize it to inspect and test

In [ ]:
ds.materialize()

Ok, but note the warning in the logs: PyArrow is not successfully parsing our JSON

In [ ]:
try:
    pa.json.read_json('/mnt/cluster_storage/ecom/users.json')
except Exception as e:
    print(e)
    

Maybe we should use JSON lines:

In [ ]:
! head /mnt/cluster_storage/ecom/users.ndjson

In [ ]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True)
ds.materialize()

No error or warning ... but no data!

How do we troubleshoot? Let's see if PyArrow reads this data properly.

In [ ]:
pa.json.read_json('/mnt/cluster_storage/ecom/users.ndjson')

In [ ]:
pd.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True)

PyArrow and Pandas are happy, so the data and libs are ok. Ray Data is filtering out non `.json` files by default

In [ ]:
ds = ray.data.read_json('/mnt/cluster_storage/ecom/users.ndjson', lines=True, file_extensions=['.ndjson'])
ds.materialize()

Given our data and our recommendation model, the plan is to factor our logic into composable parts for a pipeline:

1. read users
1. convert user IDs (GUID) to index ints for the recommender
1. infer product recommendation indexes
1. map product indexes to product records and put relevant details into the dataset
1. generate email for each record in the pipeline
1. save or enqueue (for mailing) each email

Our real-world systems will likely include databases for looking up users and products, so let's simulate a database and its API.

This example encapsulates the queries.

How will various parts of our Ray code find this database facade? We can implement it as a named Ray Actor and then look it up as needed.

In [ ]:
@ray.remote
class DatabaseFacade():
    def __init__(self, users, products):
        self.users = pd.read_json(users, lines=True)
        self.products = pd.read_parquet(products)
        
    def users_for_ids(self, ids):
        return self.users[self.users['id'].isin(ids)]
    
    def products_for_indices(self, idxs):
        return self.products.iloc[idxs]

DatabaseFacade.options(name="database").remote('/mnt/cluster_storage/ecom/users.ndjson', '/mnt/cluster_storage/ecom/cat_with_embeddings')

In [ ]:
my_db = ray.get_actor("database")

ref = my_db.users_for_ids.remote(['7034dd99-ceb3-474d-a0ba-5beaf122273f', 'cf7733df-e2db-4212-8003-69d37bd25dae'])

When calling remote methods (tasks) in a Ray program, we get `ObjectRef`s, a form of distributed pointer and promise.

When we (or Ray itself) need the referenced data, Ray can retrieve that data over the wire and deserialize to a Python object.

In [ ]:
ray.get(ref)

In [ ]:
ray.get(ref).index.values

Our conversion from user records to indices can be done with `map_batches` and a stateless function (since the state is handled by the database)

In [ ]:
def get_user_indices(batch):
    my_db = ray.get_actor("database")
    ref = my_db.users_for_ids.remote(batch['id'])
    batch['user_index'] = ray.get(ref).index.values
    return batch

In [ ]:
ds.map_batches(get_user_indices).take_batch(4)

Next, we'll load and use our recommender. In production, we'd import this but we can define it inline here to see the full code

In [ ]:
# minimal_two_tower.py
class TwoTower(nn.Module):
    def __init__(self, num_users: int, num_items: int, dim: int = 64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)

        # optional: small projection MLPs (kept minimal)
        self.user_proj = nn.Identity()
        self.item_proj = nn.Identity()

    def encode_users(self, user_ids: torch.LongTensor) -> torch.Tensor:
        u = self.user_proj(self.user_emb(user_ids))
        return F.normalize(u, dim=-1)

    def encode_items(self, item_ids: torch.LongTensor) -> torch.Tensor:
        v = self.item_proj(self.item_emb(item_ids))
        return F.normalize(v, dim=-1)

    def forward(self, user_ids: torch.LongTensor, pos_item_ids: torch.LongTensor):
        """
        Returns logits matrix [B,B] where diagonal is the positive pair and
        off-diagonals are in-batch negatives.
        """
        u = self.encode_users(user_ids)         # [B, D]
        v = self.encode_items(pos_item_ids)     # [B, D]
        logits = u @ v.t()                      # [B, B]
        return logits

Recall the batch inference pattern with Ray Data: `map_batches` using a stateful Actor class, where the Actor loads the model in its constructor.

Recall also that this pattern does not require `@ray.remote`: Ray will convert this class to an Actor for us.

In [ ]:
class Recommend():
    def __init__(self, model_location, num_recommendations, num_users, num_items):
        self.model = TwoTower(num_users, num_items)
        self.model.load_state_dict(torch.load(model_location, weights_only=True))
        self.model.eval()
        self.num_recommendations = num_recommendations
        self.num_items = num_items
        
    @torch.no_grad()
    def recommend_topk_batch(self,
        user_ids: torch.LongTensor,     # [B]
        all_item_ids: torch.LongTensor, # [N]
    ):
        """
        Returns:
          topk_indices: [B, k]  (item ids)
          topk_scores:  [B, k]
        """
        model = self.model
        k = self.num_recommendations
        model.eval()

        # Encode
        u = model.encode_users(user_ids)     # [B, D]
        v = model.encode_items(all_item_ids) # [N, D]

        # Similarity
        scores = u @ v.t()                   # [B, N]

        # Top-k per user
        topk_scores, topk_idx = torch.topk(scores, k=k, dim=1)

        # Map indices back to item ids
        topk_item_ids = all_item_ids[topk_idx]  # [B, k]

        return topk_item_ids, topk_scores
    
    def recommend_for_users(self, users):
        (items, scores) = self.recommend_topk_batch(torch.tensor(users), torch.arange(1, self.num_items))
        return items.numpy()
        
    def __call__(self, batch, column):
        users = batch[column]
        batch['recommended_items'] = self.recommend_for_users(users)
        return batch

In [ ]:
sample_batch = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']).take_batch(4)

sample_batch

In [ ]:
sample_batch['recommended_items']

Let's build up code to get some item details (name, description, and price) for each recommended item. We'll need those for the emails.

This is an interesting task because it probably makes sense to get all of the recommendations for a batch of users (i.e., num_recos_per_user * num_users) all at once from our model. 

We can use the sample batch above to develop/test/debug our code line by line before wiring it up to the Ray Data pipeline

In [ ]:
products_ref = ray.get_actor("database").products_for_indices.remote(sample_batch['recommended_items'].flatten())
products = ray.get(products_ref)
products

In [ ]:
products.iloc[0:3]

In [ ]:
products.iloc[0:3][['name', 'desc', 'price']]

In [ ]:
my_db = ray.get_actor("database")
users_in_batch = len(sample_batch['id'])
recs_per_user = sample_batch['recommended_items'].shape[1] # recos will be shape (users in batch, recos per user)
product_indices = sample_batch['recommended_items'].flatten()
ref = my_db.products_for_indices.remote(product_indices)
recommendations = ray.get(ref)
recommendations    

Next we have to split these recommendations up to match them to the respective users

In [ ]:
[recommendations[['name', 'desc', 'price']].iloc[i*recs_per_user:(i+1)*recs_per_user] for i in range(users_in_batch)]

And wrap that logic in a function

In [ ]:
def get_recommended_items_details(batch):
    my_db = ray.get_actor("database")
    users_in_batch = len(batch['id'])
    recs_per_user = batch['recommended_items'].shape[1] # recos will be shape (users in batch, recos per user)
    
    product_indices = batch['recommended_items'].flatten()
    ref = my_db.products_for_indices.remote(product_indices)
    recommendations = ray.get(ref)
    
    recs_split_by_users = [recommendations[['name', 'desc', 'price']].iloc[i*recs_per_user : (i+1)*recs_per_user] for i in range(users_in_batch)]
    batch['recommended_items_details'] = recs_split_by_users
    return batch

... for use with `map_batches`

In [ ]:
samples = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .take(2)

samples

In [ ]:
samples[0]['recommended_items_details'].iloc[0]['name']

Note the warning in the logs: the pandas DataFrame we're using for our data records is not serializing in the optimal way into a PyArrow array.

We could refactor this code to use alternate record representations if this proves to be a performance issue.

The code for generating emails doesn't trivially vectorize, so we'll start with a per-record (Ray Dataset `map`) approach.

In [ ]:
def generate_email_for_user(user):
    templated_mail = f'''Greetings, {user['first_name']} {user['last_name']}

    Check out the following personalized recommendations!

    {'; '.join([user['recommended_items_details'].iloc[i]['desc'] 
    + ' Only ' 
    + str(user['recommended_items_details'].iloc[i]['price']) for i in range(len(user['recommended_items_details']))])}

    Click here to unsubscribe.
    '''
    
    user['reco_email'] = templated_mail
    
    return user

In [ ]:
generate_email_for_user(samples[0])

In [ ]:
sample_batch = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .map(generate_email_for_user) \
    .take_batch(2)

sample_batch

We'll simulate an adapter for an email sending queue. We'll use `map_batches` to give it batches of user emails to send. The sending will be a side effect, the "output" will be a column in our dataset indicating which emails have been queued, and we'll log some queue info as a side effect as well.

In [ ]:
class EmailSendingQueue:
    def __init__(self, email_queue_service_info):
        self.queue = { 
            'service' : email_queue_service_info,
            'out_queue' : [] 
        }
    
    def __call__(self, batch):
        for email in batch['reco_email']:
            self.queue['out_queue'].append(email)
        batch['queued'] = [True] * len(batch['reco_email'])
        print(f'Currently queued {len(self.queue["out_queue"])} messages.')
        return batch

In [ ]:
sample_batch = ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .map(generate_email_for_user) \
    .map_batches(EmailSendingQueue, batch_size=32, fn_constructor_args=['sender:bulk:12.34.5678']) \
    .take_batch(2)

sample_batch

Note that although we were only trying to take a small batch, all 1000 records got processed. Why?

Our dataset is all in one block (because it's so small) and Ray Data must process an entire block.

> Check that if we repartition (to create more blocks), Ray does not process all 1000 records

This is not a big issue for us here, but this could be a big problem if we were iteratively developing code that uses an LLM where each inference takes a lot of time.

Now we're ready to generate all of our marketing emails and queue for sending

In [ ]:
ds \
    .map_batches(get_user_indices) \
    .map_batches(Recommend, fn_constructor_args=[base_model_path, 3, 1000, 1000], fn_args=['user_index']) \
    .map_batches(get_recommended_items_details) \
    .map(generate_email_for_user) \
    .map_batches(EmailSendingQueue, batch_size=32, fn_constructor_args=['sender:bulk:12.34.5678']) \
    .count()

## Online services: recommendation and search

Using Ray Serve, we'll implement the following use case first: 

* Given a user ID, we'll generate recommendations for them

In [ ]:
recommender = Recommend(base_model_path, 3, 1000, 1000)

In [ ]:
ref = my_db.users_for_ids.remote(['7034dd99-ceb3-474d-a0ba-5beaf122273f'])

In [ ]:
user_df = ray.get(ref)
user_df

In [ ]:
user_df.index.values

In [ ]:
recos = recommender.recommend_for_users(user_df.index.values)
recos

In [ ]:
ray.get(my_db.products_for_indices.remote(recos.flatten()))

We can start with a single Ray Serve logic component (a Deployment) that handles our request I/O as well as our recommendation code.

Once that works, we'll refactor to create better separation of concerns.

Here's a basic Deployment service that combines the Ingress (I/O handling+routing) role and the Recommender role.

In [ ]:
@serve.deployment()
class IngressAndRecommender:
    def __init__(self, base_model_path: str, num_recos: int, num_users: int, num_products: int):
        self.db = ray.get_actor("database")
        self.recommender = Recommend(base_model_path, num_recos, num_users, num_products)

    async def __call__(self, request: Request):  # __call__ takes a Request object
        user = await request.json()
        return self.recommend([user['id']])[['item_id', 'name', 'desc', 'price']].to_json()
    
    def recommend(self, user_ids):
        ref = self.db.users_for_ids.remote(user_ids)
        user_df = ray.get(ref)
        recos = self.recommender.recommend_for_users(user_df.index.values)
        return ray.get(self.db.products_for_indices.remote(recos.flatten()))

In [ ]:
bound_deployment = IngressAndRecommender.bind(base_model_path, 3, 1000, 1000)
app_handle = serve.run(bound_deployment)

In [ ]:
await app_handle.recommend.remote(['7034dd99-ceb3-474d-a0ba-5beaf122273f'])

In [ ]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f"})

response.json()

Since we'll be substantially re-arranging this app as we develop we can shut down everything as needed by calling `serve.shutdown`

In [ ]:
serve.shutdown()

Let's make a more single-purpose Ingress Deployment

In [ ]:
@serve.deployment()
class Ingress:
    def __init__(self, recommender):
        self.recommender = recommender

    async def __call__(self, request: Request):  # __call__ takes a Request object
        user = await request.json()
        return (await self.recommender.recommend.remote([user['id']]))[['item_id', 'name', 'desc', 'price']].to_json()

We'll make the Recommender its own Deployment (component).

To do this, the logic will stay the same but we'll need to make a minor change in how we get remote results.

* We cannot `ray.get` a response from a deployment handle (if you try, you'll get an error that says exactly that).
* Instead, we `await` the response. Similar idea, but matches async pattern for web apps.

Let's also make the DatabaseFacade into a Ray Serve Deployment -- that will make it more robust (e.g., Ray Serve will restart it if it fails) and easier to autoscale.

In [ ]:
@serve.deployment()
class Recommender:
    def __init__(self, base_model_path: str, num_recos: int, num_users: int, num_products: int, database):
        self.recommender = Recommend(base_model_path, num_recos, num_users, num_products)
        self.db = database
    
    async def recommend(self, user_ids):
        ref = self.db.users_for_ids.remote(user_ids)
        user_df = await ref
        recos = self.recommender.recommend_for_users(user_df.index.values)
        return await self.db.products_for_indices.remote(recos.flatten())

In [ ]:
@serve.deployment()
class DatabaseFacade():
    def __init__(self, users, products):
        self.users = pd.read_json(users, lines=True)
        self.products = pd.read_parquet(products)
        
    def users_for_ids(self, ids):
        return self.users[self.users['id'].isin(ids)]
    
    def products_for_indices(self, idxs):
        return self.products.iloc[idxs]
    
    def all_products(self):
        return self.products

Now we have to create the bound deployments in dependency order

In [ ]:
bound_db_facade_deployment = DatabaseFacade.bind('/mnt/cluster_storage/ecom/users.ndjson', '/mnt/cluster_storage/ecom/cat_with_embeddings')

bound_rec_deployment = Recommender.bind(base_model_path, 3, 1000, 1000, bound_db_facade_deployment)

bound_ingress = Ingress.bind(bound_rec_deployment)

And run

In [ ]:
app_handle = serve.run(bound_ingress)

In [ ]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f"})

response.json()

Finally, let's implement a semantic search using our product description embeddings. The search component will also be a Deployment.

In [ ]:
@serve.deployment()
class SemanticSearch():
    async def __init__(self, model, db):
        self.model = SentenceTransformer(model)
        self.db = db
        self.all_products_embeddings = (await self.db.all_products.remote())['desc_emb']
        
    async def search(self, query, matches):
        similarities = self.model.similarity(self.model.encode(query), self.all_products_embeddings)
        top_matches = similarities.flatten().topk(50).indices
        products = await self.db.products_for_indices.remote(np.array(top_matches))
        # rerank
        top_name_distances = torch.tensor([textdistance.lcsstr.similarity(query, prodname) for prodname in list(products['name'])]).topk(matches).indices
        results = products.iloc[np.array(top_name_distances)]
        
        return results

In [ ]:
cached_embedding_model = "/mnt/cluster_storage/ecom/hf_cache/models--google--embeddinggemma-300m/snapshots/57c266a740f537b4dc058e1b0cda161fd15afa75"

bound_search = SemanticSearch.bind(cached_embedding_model, bound_db_facade_deployment)

We'll re-write the Ingress Deployment to support routing: if the service is called with a query, we'll route to that Deployment.

In [ ]:
@serve.deployment()
class Ingress:
    def __init__(self, recommender, search):
        self.recommender = recommender
        self.search = search

    async def __call__(self, request: Request):  # __call__ takes a Request object
        user = await request.json()
        recommendations = (await self.recommender.recommend.remote([user['id']]))[['item_id', 'name', 'desc', 'price']]
        result = { "recommendations" : recommendations.to_json() }
        if "query" in user:
            search_results = (await self.search.search.remote(user['query'], 5))[['item_id', 'name', 'desc', 'price']]
            result["search_results"] = search_results.to_json()
        
        return json.dumps(result)

In [ ]:
bound_ingress = Ingress.bind(bound_rec_deployment, bound_search)

In [ ]:
app_handle = serve.run(bound_ingress)

In [ ]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f", "query" : "steel bols"})

response.json()

In [ ]:
response = requests.post("http://localhost:8000/", json={ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f" })

response.json()

In [ ]:
! serve shutdown -y

## Deploying via the CLI

Inspect the refactored code in 
* `recommend.py`
* `search_and_recommend.py`

Pay extra attention to...
* file paths that will be available/visible at runtime
* imported code (and working dir)
* environment vars and other requirements
* ambiguous names (scoping and namespacing isn't perfect since Python is not yet designed for a distributed runtime)

In [ ]:
! serve build search_and_recommend:bound_ingress -o serve_config.yaml

In [ ]:
! cat serve_config.yaml

We'll update this to produce `serve_updated.yaml`

(reference https://docs.ray.io/en/latest/serve/production-guide/config.html)

```yaml
# This file was generated using the `serve build` command on Ray v2.55.1.

proxy_location: EveryNode

http_options:
  host: 0.0.0.0
  port: 8000

grpc_options:
  port: 9000
  grpc_servicer_functions: []

logging_config:
  encoding: JSON
  log_level: INFO
  logs_dir: null
  enable_access_log: true
  additional_log_standard_attrs: []

applications:
- name: search_recommend
  route_prefix: /
  import_path: search_and_recommend:bound_ingress
  runtime_env:
    working_dir: "https://anyscale-public-materials-use2.s3.us-east-2.amazonaws.com/recommend.zip"
  deployments:
  - name: DatabaseFacade
    num_replicas: 4
    graceful_shutdown_wait_loop_s: 2.0
    graceful_shutdown_timeout_s: 20.0
    health_check_period_s: 10.0
    health_check_timeout_s: 30.0
        
  - name: Recommender
    num_replicas: 2
    max_ongoing_requests: 100
    ray_actor_options:
      num_cpus: 1
      num_gpus: 0.5
      
  - name: SemanticSearch
    autoscaling_config:
      min_replicas: 2
      max_replicas: 4

  - name: Ingress
    autoscaling_config:
      min_replicas: 2
      max_replicas: 4
```

We'll deploy this service from the CLI

In [ ]:
! serve deploy serve_updated.yaml

In [ ]:
response = requests.post("http://localhost:8000/", json='{ "id" : "7034dd99-ceb3-474d-a0ba-5beaf122273f", "query" : "set of bowls"}')

response.json()

In [ ]:
! serve shutdown -y